# NutriAssist-RW: fine-tuning a bilingual child-nutrition assistant with QLoRA

**Problem.** Caregivers of children under 5 in Rwanda need practical, safe nutrition advice in the language they speak. NutriAssist-RW answers questions on breastfeeding, feeding children from 6 months to 5 years, maternal nutrition and local foods, in **English and Kinyarwanda**, and sends caregivers to the health centre when there are danger signs.

**Approach.** We fine-tune a small open instruction model (**Qwen2.5-1.5B-Instruct**) on 513 curated question-answer pairs, using **QLoRA**: the base model is frozen in 4-bit precision and only small low-rank adapter matrices are trained. This fits on a free Colab T4 GPU.

| Section | What it shows |
|---|---|
| 1. Setup | Installs the `nutriassist` package from GitHub and connects Google Drive |
| 2. Data | Data engineering steps, distributions, token lengths, nutrient correlations |
| 3. Model architecture | Layers, activation, normalisation, attention, LoRA placement, optimisation |
| 4. Baseline | The base model's scores before fine-tuning |
| 5. Experiments | 6 hyperparameter runs with training time and peak GPU memory |
| 6. Results | Test-set BLEU, chrF, ROUGE-L, perplexity, accuracy, precision, recall, F1; improvement over the base model |
| 7. Qualitative | Base and fine-tuned answers side by side |
| 8. Demo and export | A Gradio chat demo; the model exported for the CPU web app, tested, and how to download it |

**How to run.**
- *In the browser (recommended for the long training run):* *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- *In VS Code (Google Colab extension):* *Select Kernel → Colab → New Colab Server → GPU → T4*. Then *Ctrl+Shift+P → "Colab: Mount Google Drive to Server..."* and run the cell it adds, then *Run All*.
**If Colab disconnects, nothing finished is lost.** Everything is saved to `MyDrive/NutriAssist-RW/outputs/` as it happens:

| Step | What is saved | After a disconnect |
|---|---|---|
| Training | a checkpoint every 10 steps (about every minute) | continues from the last checkpoint |
| A finished training run | the LoRA adapter | is never trained again |
| Generating answers for evaluation | every 16 answers | only the missing answers are generated |
| Each finished experiment, the baseline, the final test | scores and predictions | loaded from Drive |

Reconnect, then *Runtime → Run all* again. The **Progress so far** cell (section 1) shows what is already done.

**Avoiding disconnects (free Colab).** Run it in the browser, not in VS Code, and keep the tab open and visible. Free Colab disconnects after about 90 minutes without any interaction, so click in the notebook now and then. Colab also limits free GPU time per day: if it says it cannot connect to a GPU, wait a few hours (the limit resets) and continue; the notebook picks up where it stopped.

All the code lives in the tested `nutriassist` package in the repository (`src/nutriassist/`); this notebook calls it and presents the results.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/pauline12ish34/NutriAssist_Capstone.git"
REPO_DIR = Path("/content/NutriAssist_Capstone")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" gradio

sys.path.insert(0, str(REPO_DIR / "src"))  # an editable install is only picked up after a restart
os.chdir(REPO_DIR)
!git log -1 --format="Code version: %h %s"

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as error:
        # The VS Code Colab extension cannot run drive.mount(); it mounts Drive with its own command.
        raise RuntimeError(
            "Could not mount Google Drive. In VS Code: press Ctrl+Shift+P, run "
            "'Colab: Mount Google Drive to Server...', run the cell it adds, then run this cell again."
        ) from error

OUTPUT_DIR = DRIVE_ROOT / "NutriAssist-RW" / "outputs"
PRED_DIR = OUTPUT_DIR / "predictions"
ADAPTERS_DIR = OUTPUT_DIR / "adapters"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"   # on Drive so training can resume after a disconnect
for folder in (OUTPUT_DIR, PRED_DIR, ADAPTERS_DIR, CHECKPOINTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Outputs are saved to", OUTPUT_DIR)

In [ ]:
import torch

cuda_available = torch.cuda.is_available()

print("CUDA available:", cuda_available)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU connected")

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import seaborn as sns
import torch
import transformers

from nutriassist.evaluation import EvaluationResult, Evaluator, compare
from nutriassist.inference.transformers_model import TransformersChatModel, load_causal_lm, load_tokenizer
from nutriassist.training import ChatEncoder, QADataset, TrainingConfig
from nutriassist.training.architecture import architecture_summary, lora_target_modules
from nutriassist.training.experiments import ExperimentRunner
from nutriassist.training.finetune import free_gpu_memory

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then run all again.")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 1024**3:.1f} GB")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_colwidth", 120)

### Progress so far

What is already saved on Drive. Everything marked done is loaded, not repeated.

In [ ]:
def progress_report() -> pd.DataFrame:
    rows = [{"step": f"baseline / {split}", "status": "done" if (PRED_DIR / f"base_{split}_scores.json").exists() else "to do"}
            for split in ("val", "test", "paraphrases")]
    finished = set(pd.read_csv(OUTPUT_DIR / "experiments.csv")["experiment"]) if (OUTPUT_DIR / "experiments.csv").exists() else set()
    for name in ["exp1_baseline", "exp2_lr_1e-4", "exp3_lr_5e-4", "exp4_rank_8", "exp5_rank_32", "exp6_epochs_5"]:
        if name in finished:
            status = "done"
        elif (ADAPTERS_DIR / name / "training_summary.json").exists():
            status = "trained, validation scores to do"
        elif (CHECKPOINTS_DIR / name).exists():
            checkpoints = sorted((CHECKPOINTS_DIR / name).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
            status = f"training started, will resume from {checkpoints[-1].name}" if checkpoints else "training started"
        else:
            status = "to do"
        rows.append({"step": f"experiment {name}", "status": status})
    final = sorted(PRED_DIR.glob("final_*_test_scores.json"))
    rows.append({"step": "final test evaluation", "status": "done" if final else "to do"})
    return pd.DataFrame(rows).set_index("step")


progress_report()

## 2. Data

### 2.1 Data engineering

The data was prepared in the repository by two pipelines (code in `src/nutriassist/`, tests in `tests/`):

**Food table** (`food_data`, details in `docs/nutrient_cleaning.md`). The Kaggle food workbook (2,395 foods, 5 sheets) stores values **per typical serving, not per 100 g**, records sodium in grams and some vitamins in mg instead of µg, and in 300 rows the micronutrient columns are shifted by one. The pipeline estimates each serving's mass, rescales to per 100 g, fixes the units and the column shift, and flags every value (`ok`, `below_precision`, `zero_or_missing`, `lost_in_shift`).

**QA dataset** (`qa_data`, details in `docs/qa_dataset.md`).
1. The 135 original pairs were triaged: kept, edited for safety, rewritten as a referral, or dropped (23 dropped, e.g. adult malnutrition or IV protocols that only clinicians should follow).
2. Curated pairs were added for the narrowed scope (caregivers of children 0–59 months, and maternal nutrition), with a Kinyarwanda version of every question, reviewed and approved by a native speaker.
3. Nutrient questions were generated from the cleaned food table, using only foods whose values were checked against USDA.
4. Pairs were split 75/10/15 into train/val/test **by group**: a question, its paraphrases and its translation always land in the same split, so test questions never leak into training. The split is stratified by category.
5. Paraphrases of training questions are added to training. Paraphrases of val/test questions are kept apart as a robustness test.
6. Every example is stored in chat format: the system prompt, the question and the answer.

In [ ]:
dataset = QADataset.load(REPO_DIR / "data/QA_data/processed")
master = pd.read_json(REPO_DIR / "data/QA_data/processed/qa_master.jsonl", lines=True)
print(f"{len(master)} unique question-answer pairs; chat examples per split:")
display(dataset.summary())

example = dataset.test[0]
print(f"\nExample ({example.lang}, {example.category})\nQ: {example.question}\nA: {example.answer}")

### 2.2 Distributions

How the pairs spread across topics and languages, and how the stratified split keeps the same topic mix in train, val and test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = master["category"].value_counts().index
sns.countplot(data=master, y="category", hue="lang", order=order, ax=axes[0])
axes[0].set(title="QA pairs per category and language", xlabel="pairs", ylabel="")

split_mix = pd.crosstab(master["category"], master["split"], normalize="columns").loc[order, ["train", "val", "test"]]
sns.heatmap(split_mix * 100, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of split"}, ax=axes[1])
axes[1].set(title="Category share within each split (%)", xlabel="", ylabel="")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "data_categories.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
master["source"].value_counts().plot.barh(ax=axes[0], title="Where the pairs come from")
axes[0].set_xlabel("pairs")
frame = dataset.to_frame().query("split != 'paraphrases'")
sns.histplot(data=frame, x="answer_words", hue="lang", bins=25, multiple="dodge", ax=axes[1])
axes[1].set(title="Answer length", xlabel="words per answer")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "data_sources_lengths.png", dpi=150)
plt.show()
display(frame.groupby("lang")[["question_words", "answer_words"]].describe().round(1))

### 2.3 Token lengths

The model reads tokens, not words. Kinyarwanda is rare in the tokenizer's training data, so the same answer takes more tokens in Kinyarwanda than in English. The training limit (`max_length`) must fit the longest example, or answers get cut off.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = TrainingConfig().max_length

tokenizer = load_tokenizer(BASE_MODEL)
encoder = ChatEncoder(tokenizer, MAX_LENGTH)
token_frame = pd.DataFrame({
    "lang": [ex.lang for ex in dataset.train],
    "answer_tokens": [len(tokenizer(ex.answer, add_special_tokens=False)["input_ids"]) for ex in dataset.train],
    "answer_words": [len(ex.answer.split()) for ex in dataset.train],
    "total_tokens": [len(encoder.encode(ex)["input_ids"]) for ex in dataset.train],
})
token_frame["tokens_per_word"] = token_frame["answer_tokens"] / token_frame["answer_words"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(data=token_frame, x="total_tokens", hue="lang", bins=30, ax=axes[0])
axes[0].axvline(MAX_LENGTH, color="red", linestyle="--", label=f"max_length = {MAX_LENGTH}")
axes[0].legend()
axes[0].set(title="Tokens per training example (prompt + answer)", xlabel="tokens")
sns.boxplot(data=token_frame, x="lang", y="tokens_per_word", ax=axes[1])
axes[1].set(title="Tokens per word of answer", xlabel="language")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "data_tokens.png", dpi=150)
plt.show()
_, stats = encoder.encode_many(dataset.train)
print(stats)
display(token_frame.groupby("lang")[["answer_tokens", "tokens_per_word"]].median().round(2))

### 2.4 Nutrient correlations in the food table

The nutrient answers and the meal planner use the cleaned food table. The heatmap shows Spearman (rank) correlations between nutrients per 100 g, using only values flagged `ok`. Rank correlation is used because nutrient amounts are very skewed (a few foods are extremely rich in one nutrient).

In [ ]:
foods = pd.read_csv(REPO_DIR / "data/processed/foods.csv")
nutrients = pd.read_csv(REPO_DIR / "data/processed/food_nutrients.csv")
reliable = foods.loc[foods["basis_confidence"] == "high", "food_id"]
values = (nutrients[nutrients["food_id"].isin(reliable) & (nutrients["value_quality"] == "ok")]
          .pivot_table(index="food_id", columns="nutrient_name", values="amount_per_100g"))
KEY_NUTRIENTS = ["caloric_value", "protein", "fat", "carbohydrates", "dietary_fiber", "sugars", "water",
                 "iron", "zinc", "calcium", "vitamin_a", "vitamin_c"]
corr = values[KEY_NUTRIENTS].corr(method="spearman", min_periods=50)

plt.figure(figsize=(10, 8))
ax = sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True)
ax.grid(False)  # blank cells: fewer than 50 foods have reliable values for both nutrients
plt.title(f"Spearman correlation of nutrients per 100 g ({len(values)} foods with reliable values)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nutrient_correlations.png", dpi=150)
plt.show()

## 3. Model architecture

Qwen2.5-1.5B-Instruct is a **decoder-only transformer**: the question is split into tokens, each token becomes a vector (embedding), and the vectors pass through a stack of identical decoder blocks. The model then predicts the next token, one at a time, until the answer is complete.

Each decoder block has:
- **RMSNorm** normalisation before each sub-layer, and residual (skip) connections around them;
- **grouped-query self-attention** with **rotary position embeddings (RoPE)**: several query heads share each key/value head, which saves memory;
- a **SwiGLU MLP**: `down( SiLU(gate(x)) * up(x) )`, so the activation function is **SiLU** (also called Swish).

The output layer maps the final vector to a score for every token in the vocabulary, and **softmax** turns the scores into probabilities.

**How QLoRA trains it.**
- The base weights are loaded in **4-bit NF4** with double quantisation and stay **frozen**.
- A **LoRA adapter** is added to every linear layer (attention q, k, v, o and MLP gate, up, down). The adapter adds `(alpha / r) · B·A` to the frozen weight, where A and B are small matrices of rank r. Only A and B are trained, which is about 1–2% of the parameters.
- **Loss:** cross-entropy on the answer tokens only. The prompt tokens are masked out.
- **Optimiser:** paged AdamW with 8-bit states, a cosine learning-rate schedule with linear warm-up, fp16 mixed precision, and gradient checkpointing to save memory.

In [ ]:
from peft import LoraConfig, get_peft_model

model = load_causal_lm(BASE_MODEL, quantize_4bit=True)
display(pd.Series(architecture_summary(model.config), name=BASE_MODEL).to_frame())
print(f"Memory footprint in 4-bit: {model.get_memory_footprint() / 1024**3:.2f} GB")

default = TrainingConfig()
model = get_peft_model(model, LoraConfig(r=default.lora_rank, lora_alpha=default.lora_alpha,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
trainable, total = model.get_nb_trainable_parameters()
print(f"LoRA (rank {default.lora_rank}, alpha {default.lora_alpha}) on: {', '.join(lora_target_modules(n for n, _ in model.named_modules()))}")
print(f"Trainable parameters: {trainable / 1e6:.1f} M of {total / 1e6:.0f} M ({100 * trainable / total:.2f}%)")

first_block = next(module for name, module in model.named_modules() if name.endswith("layers.0"))
print("\nOne decoder block, with its LoRA adapters:\n")
print(first_block)

In [ ]:
def draw_architecture(summary: dict, lora_targets: list[str]) -> None:
    """Block diagram of the decoder-only transformer with LoRA, labelled from the model config."""
    layers = summary.get("Decoder layers (transformer blocks)", "N")
    hidden = summary.get("Hidden size (width of each token vector)", "?")
    heads = summary.get("Attention heads (queries)", "?")
    kv_heads = summary.get("Key/value heads (grouped-query attention)", "?")
    mlp = summary.get("MLP inner size", "?")
    vocab = summary.get("Vocabulary size (tokens)", "?")
    act = summary.get("MLP activation function", "silu")
    boxes = [
        ("Input tokens (system prompt + question)", "#e8eef7"),
        (f"Token embedding  ({vocab:,} tokens → {hidden}-d)" if isinstance(vocab, int) else "Token embedding", "#e8eef7"),
        ("RMSNorm", "#fdf2d0"),
        (f"Grouped-query self-attention + RoPE\n{heads} query heads, {kv_heads} key/value heads\nLoRA on q, k, v, o", "#fbe0d8"),
        ("+ residual", "#ffffff"),
        ("RMSNorm", "#fdf2d0"),
        (f"SwiGLU MLP ({hidden} → {mlp} → {hidden}), activation {act.upper()}\nLoRA on gate, up, down", "#fbe0d8"),
        ("+ residual", "#ffffff"),
        ("Final RMSNorm", "#fdf2d0"),
        ("LM head → softmax → next-token probabilities", "#e3f1e3"),
    ]
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(boxes) + 1)
    ax.axis("off")
    for i, (text, color) in enumerate(boxes):
        y = len(boxes) - i
        ax.text(5, y, text, ha="center", va="center", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.5", facecolor=color, edgecolor="#555555"))
        if i < len(boxes) - 1:
            ax.annotate("", xy=(5, y - 0.62), xytext=(5, y - 0.38), arrowprops=dict(arrowstyle="->", color="#555555"))
    block_top, block_bottom = len(boxes) - 2 + 0.45, len(boxes) - 7 - 0.45
    ax.add_patch(plt.Rectangle((0.4, block_bottom), 9.2, block_top - block_bottom, fill=False,
                               linestyle="--", edgecolor="#1f4e79", linewidth=1.5))
    ax.text(0.5, block_top - 0.15, f"× {layers} decoder blocks", color="#1f4e79", fontsize=11, fontweight="bold", va="top")
    ax.set_title(f"{BASE_MODEL} with QLoRA adapters ({', '.join(lora_targets)})", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "model_architecture.png", dpi=150)
    plt.show()


draw_architecture(architecture_summary(model.config), lora_target_modules(n for n, _ in model.named_modules()))
del model
free_gpu_memory()

## 4. Baseline: the base model before fine-tuning

The base model gets the same system prompt and questions as the fine-tuned model will. Its scores are the "before" numbers.

**Metrics.**
- **BLEU** and **ROUGE-L**: word overlap with the reference answer (0–100, higher is better).
- **chrF**: character n-gram overlap. It is fairer than BLEU for Kinyarwanda, where one word carries many prefixes and suffixes.
- **Perplexity**: how surprised the model is by the reference answer (lower is better).
- **Language match**: % of answers written in the language of the question.
- **Referral accuracy, precision, recall and F1**: the safety behaviour as a yes/no classification. Does the answer send the caregiver to a health centre or health worker when the reference answer does, and not otherwise?
- **Danger referral rate**: % of danger-sign questions (medicines, doses, severe symptoms) answered with a referral.

Answers are generated greedily (no sampling), so every score is reproducible.

In [ ]:
evaluator = Evaluator()
EVAL_SPLITS = ("val", "test", "paraphrases")


def evaluate_cached(tag: str, load_model, splits=EVAL_SPLITS) -> dict[str, EvaluationResult]:
    """Evaluate on each split, or load the saved result from Drive when it already exists."""
    missing = [s for s in splits if not EvaluationResult.exists(PRED_DIR, f"{tag}_{s}")]
    if missing:
        chat_model = load_model()
        for split in missing:
            start = time.perf_counter()
            # Answers are saved every 16 questions, so a disconnect does not lose the ones already generated.
            result = evaluator.evaluate(chat_model, dataset.splits()[split],
                                        cache_path=PRED_DIR / f"{tag}_{split}_answers.jsonl")
            result.save(PRED_DIR, f"{tag}_{split}")
            print(f"{tag} / {split}: {(time.perf_counter() - start) / 60:.1f} min")
        del chat_model
        free_gpu_memory()
    return {s: EvaluationResult.load(PRED_DIR, f"{tag}_{s}") for s in splits}


base = evaluate_cached("base", lambda: TransformersChatModel.from_pretrained(BASE_MODEL))
display(base["test"].scores_frame().round(2))

## 5. Hyperparameter experiments

Every experiment changes **one** setting from `exp1_baseline`, so each row of the table shows the effect of that one setting:

| Experiment | Change | Question it answers |
|---|---|---|
| exp1_baseline | lr 2e-4, rank 16, 3 epochs | Reference point (common QLoRA defaults) |
| exp2_lr_1e-4 / exp3_lr_5e-4 | learning rate | Is learning too slow, or unstable? |
| exp4_rank_8 / exp5_rank_32 | LoRA rank | How much adapter capacity does the task need? |
| exp6_epochs_5 | epochs | Does more training help, or does it overfit 513 pairs? |

Each run records training time, peak GPU memory, losses and validation scores. **The test set is not used here**: choosing settings on test scores would make the final result look better than it really is.

This is the longest step (about 10–15 minutes per experiment on a T4). It can be split over several Colab sessions: training resumes from its last checkpoint, and the `sessions` column of the table shows how many sessions a run took. Training time and peak GPU memory are added up across sessions.

In [ ]:
baseline_config = TrainingConfig(name="exp1_baseline", base_model=BASE_MODEL)
EXPERIMENTS = [
    baseline_config,
    baseline_config.variant("exp2_lr_1e-4", learning_rate=1e-4),
    baseline_config.variant("exp3_lr_5e-4", learning_rate=5e-4),
    baseline_config.variant("exp4_rank_8", lora_rank=8),
    baseline_config.variant("exp5_rank_32", lora_rank=32),
    baseline_config.variant("exp6_epochs_5", epochs=5),
]
pd.DataFrame([c.to_dict() for c in EXPERIMENTS]).set_index("name")

In [ ]:
runner = ExperimentRunner(dataset, OUTPUT_DIR, evaluator, checkpoints_dir=CHECKPOINTS_DIR)
experiments = runner.run(EXPERIMENTS)

### 5.1 Experiment table

The first row is the base model without fine-tuning. The best value in each validation column is highlighted.

In [ ]:
base_val = {f"val_{k}": v for k, v in base["val"].scores["all"].items()}
table = pd.concat([pd.DataFrame([{"experiment": "base model (no fine-tuning)", **base_val}]), experiments], ignore_index=True)
COLUMNS = ["experiment", "learning_rate", "lora_rank", "epochs", "trainable_params_M", "train_minutes", "sessions",
           "peak_gpu_reserved_GB", "train_loss", "val_loss", "val_perplexity", "val_BLEU", "val_chrF",
           "val_ROUGE-L", "val_language_match_%", "val_referral_f1_%"]
table[COLUMNS].to_csv(OUTPUT_DIR / "experiment_table.csv", index=False)
(table[COLUMNS].style
 .highlight_max(subset=["val_BLEU", "val_chrF", "val_ROUGE-L", "val_referral_f1_%"], color="#c6efce")
 .highlight_min(subset=["val_loss", "val_perplexity"], color="#c6efce")
 .format(precision=3, na_rep="–"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for name in experiments["experiment"]:
    history = json.loads((ADAPTERS_DIR / name / "log_history.json").read_text())
    axes[0].plot([h["epoch"] for h in history if "loss" in h], [h["loss"] for h in history if "loss" in h], label=name)
    axes[1].plot([h["epoch"] for h in history if "eval_loss" in h], [h["eval_loss"] for h in history if "eval_loss" in h],
                 marker="o", label=name)
axes[0].set(title="Training loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].legend(fontsize=8)
cost = experiments.set_index("experiment")[["train_minutes", "peak_gpu_reserved_GB"]]
cost.columns = ["training time (min)", "peak GPU memory (GB)"]
cost.plot.bar(ax=axes[2], rot=30, title="Cost per experiment")
axes[2].set(xlabel="", ylabel="minutes / GB")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "experiment_curves.png", dpi=150)
plt.show()

### 5.2 Choosing the final model

The final model is the experiment with the highest **validation ROUGE-L**; the lowest validation loss breaks ties. To choose differently (for example a cheaper run with almost the same score), set `BEST_EXPERIMENT` by name.

In [ ]:
BEST_EXPERIMENT = None
BEST_EXPERIMENT = BEST_EXPERIMENT or runner.best("val_ROUGE-L", "val_loss")
best = experiments.set_index("experiment").loc[BEST_EXPERIMENT]
print(f"Final model: {BEST_EXPERIMENT}")
display(best.to_frame().T)

## 6. Results on the test set

This is the first time the test questions are answered by a fine-tuned model. The held-out paraphrases are reworded val/test questions the model never saw, so they show whether it learned the content or only memorised question wording.

In [ ]:
tuned = evaluate_cached(
    f"final_{BEST_EXPERIMENT}",
    lambda: TransformersChatModel.from_pretrained(best["base_model"], ADAPTERS_DIR / BEST_EXPERIMENT),
    splits=("test", "paraphrases"),
)
comparison = pd.concat([compare(base[s], tuned[s]).assign(split=s) for s in ("test", "paraphrases")], ignore_index=True)
comparison.to_csv(OUTPUT_DIR / "final_comparison.csv", index=False)

test_table = comparison[comparison["split"] == "test"].pivot(index="metric", columns="group",
                                                              values=["base", "fine_tuned", "improvement_%"])
display(test_table.round(1))

### 6.1 Did fine-tuning improve the model by at least 10%?

In [ ]:
headline = comparison.query("split == 'test' and group == 'all'").set_index("metric")
for metric in ["ROUGE-L", "chrF", "BLEU", "perplexity", "language_match_%", "referral_f1_%"]:
    row = headline.loc[metric]
    verdict = "meets the 10% target" if row["improvement_%"] >= 10 else "below 10%"
    print(f"{metric:>18}: {row['base']:7.2f} -> {row['fine_tuned']:7.2f}   {row['improvement_%']:+6.1f}%   {verdict}")

fig, ax = plt.subplots(figsize=(11, 4))
plot_metrics = ["BLEU", "chrF", "ROUGE-L", "language_match_%", "referral_f1_%"]
scores = (comparison.query("split == 'test' and group != 'all' and metric in @plot_metrics")
          .melt(id_vars=["group", "metric"], value_vars=["base", "fine_tuned"], var_name="model"))
scores["model / language"] = scores["model"] + " / " + scores["group"]
sns.barplot(data=scores, x="metric", y="value", hue="model / language", ax=ax)
ax.set(title="Test set: base vs fine-tuned, per language", xlabel="", ylabel="score (0–100)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_scores.png", dpi=150)
plt.show()

### 6.2 Safety behaviour as a classification

Rows are what the reference answer does, columns what the model does. A good model stays on the diagonal: it refers when it should, and gives practical advice otherwise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, result) in zip(axes, [("Base model", base["test"]), (f"Fine-tuned ({BEST_EXPERIMENT})", tuned["test"])]):
    matrix = pd.crosstab(result.predictions["reference_refers"], result.predictions["prediction_refers"]) \
        .reindex(index=[False, True], columns=[False, True], fill_value=0)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["no referral", "referral"], yticklabels=["no referral", "referral"])
    ax.set(title=title, xlabel="model answer", ylabel="reference answer")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "referral_confusion.png", dpi=150)
plt.show()
display(headline.loc[[m for m in headline.index if m.startswith(("referral", "danger"))]].round(1))

### 6.3 Robustness to rewording

In [ ]:
robustness = comparison.query("group == 'all' and metric in ['ROUGE-L', 'chrF', 'BLEU']") \
    .pivot(index="metric", columns="split", values="fine_tuned").round(2)
robustness["drop_on_paraphrases_%"] = (100 * (robustness["test"] - robustness["paraphrases"]) / robustness["test"]).round(1)
display(robustness)

## 7. Qualitative comparison

Scores do not show whether an answer is safe, practical and natural. Below are the same test questions answered by the base model and the fine-tuned model, one per language and category. All answers are in `outputs/qualitative_test.csv`.

In [ ]:
side_by_side = base["test"].predictions[["id", "lang", "category", "question", "reference", "prediction"]] \
    .rename(columns={"prediction": "base_answer"}) \
    .merge(tuned["test"].predictions[["id", "prediction"]].rename(columns={"prediction": "fine_tuned_answer"}), on="id")
side_by_side.to_csv(OUTPUT_DIR / "qualitative_test.csv", index=False)

sample = side_by_side.groupby(["lang", "category"]).head(1).groupby("lang").head(4)
for _, row in sample.iterrows():
    print("=" * 110)
    print(f"[{row['lang']} | {row['category']}]  {row['question']}\n")
    print(f"REFERENCE:  {row['reference']}\n")
    print(f"BASE:       {row['base_answer']}\n")
    print(f"FINE-TUNED: {row['fine_tuned_answer']}\n")

## 8. Demo and export

### 8.1 Try the model (Gradio)

Run the cell and open the public link. Ask in English or Kinyarwanda. The full web app (FastAPI backend and Streamlit frontend) is in the repository's `app/` folder.

In [ ]:
import gradio as gr

from nutriassist.prompts import build_messages

chat_model = TransformersChatModel.from_pretrained(best["base_model"], ADAPTERS_DIR / BEST_EXPERIMENT)

demo = gr.ChatInterface(
    fn=lambda message, history: chat_model.generate(build_messages(message)),
    title="NutriAssist-RW",
    description="Nutrition advice for caregivers of children under 5 in Rwanda, in English or Kinyarwanda. "
                "Not a substitute for a health worker: for danger signs, go to the health centre.",
    examples=["What foods should I give my 8-month-old baby?", "Umwana w'amezi 9 yarya iki?",
              "My child has diarrhoea and is very weak. What should I do?"],
)
demo.launch(share=True)

### 8.2 Export the model for the web app

The web app runs on ordinary computers without a GPU, so we:
1. **merge** the LoRA adapter into the base weights, giving one standalone model;
2. convert it to **GGUF**, the format `llama.cpp` uses for fast CPU inference;
3. save two sizes to Drive:

| File | Size | Use |
|---|---|---|
| `nutriassist-rw-q4_k_m.gguf` | about 1.0 GB | **recommended**: faster on a laptop CPU and quicker to download; small quality loss |
| `nutriassist-rw-q8_0.gguf` | about 1.6 GB | highest quality; for a stronger computer or the online deployment |

Both are made from a full-precision (16-bit) conversion, which gives better quality than shrinking the 8-bit file. If the session disconnects, run the cell again: files that already exist are kept.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

if "demo" in globals():
    demo.close()
chat_model = None
free_gpu_memory()

MERGED_DIR = Path("/content/nutriassist-merged")
F16_PATH = Path("/content/nutriassist-rw-f16.gguf")      # temporary (3 GB), not saved to Drive
GGUF_FILES = {kind: OUTPUT_DIR / f"nutriassist-rw-{kind}.gguf" for kind in ("q4_k_m", "q8_0")}

if not all(path.exists() for path in GGUF_FILES.values()):
    if not F16_PATH.exists():
        full_precision = AutoModelForCausalLM.from_pretrained(best["base_model"], dtype=torch.float16, device_map={"": 0})
        merged = PeftModel.from_pretrained(full_precision, ADAPTERS_DIR / BEST_EXPERIMENT).merge_and_unload()
        merged.save_pretrained(MERGED_DIR)
        load_tokenizer(ADAPTERS_DIR / BEST_EXPERIMENT).save_pretrained(MERGED_DIR)
        del merged, full_precision
        free_gpu_memory()
        if not Path("/content/llama.cpp").exists():
            !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
        !pip install -q /content/llama.cpp/gguf-py
        !python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16_PATH} --outtype f16

    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
    from nutriassist.inference.quantize import quantize_gguf

    for kind, path in GGUF_FILES.items():
        if not path.exists():
            print(f"Making {path.name}...")
            quantize_gguf(F16_PATH, path, kind)

for path in GGUF_FILES.values():
    print(f"{path.name}: {path.stat().st_size / 1024**3:.2f} GB")

### 8.3 Check the exported model on a CPU

The web app will run the GGUF file on a CPU, so test exactly that here: the smaller model answers through the same safety pipeline as the app. Colab's CPU has only 2 cores, so each answer takes a while; a laptop is usually faster.

In [ ]:
from nutriassist.assistant import NutritionAssistant
from nutriassist.inference.llamacpp_model import LlamaCppChatModel

cpu_model = LlamaCppChatModel.from_file(GGUF_FILES["q4_k_m"])
cpu_assistant = NutritionAssistant(cpu_model)
for question in ["What foods should I give my 8-month-old baby?",
                 "Umwana w'amezi 9 yarya iki?",
                 "My child is having a seizure. What should I do?"]:
    start = time.perf_counter()
    reply = cpu_assistant.ask(question)
    print(f"Q: {question}\n[{reply.source.value}, {time.perf_counter() - start:.1f} s]\n{reply.answer}\n")
del cpu_assistant, cpu_model

### 8.4 Download the model to your computer

1. Open https://drive.google.com, then *My Drive → NutriAssist-RW → outputs*.
2. Right-click **`nutriassist-rw-q4_k_m.gguf`** → *Download*. Google says it cannot scan large files for viruses: click *Download anyway*.
   On a slow connection, if the download stops, open Chrome's downloads page (Ctrl+J) and click *Resume*.
3. Move the file into the **`models/`** folder of the project on your computer (create the folder if it is missing).
   Git ignores this folder: the file is too large for GitHub (limit 100 MB), so it stays on your computer.
4. Start the app with `python scripts/run_local.py`. The sidebar should say *Assistant is online* and **not** *Demo mode*.

Also download these small files for the report and the README: `experiment_table.csv`, `final_comparison.csv`, `qualitative_test.csv` and the `.png` figures.

### 8.5 Optional: publish to Hugging Face Hub

Needed for the online deployment (see `docs/deployment.md`): the Hugging Face Space downloads the model from here. Set `HF_REPO` (for example `"your-username/nutriassist-rw"`) and run the cell; you will be asked for a Hugging Face **write** token. Uploading from Colab is much faster than uploading from your own connection.

In [ ]:
HF_REPO = None

if HF_REPO:
    from huggingface_hub import HfApi, notebook_login

    notebook_login()
    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(ADAPTERS_DIR / BEST_EXPERIMENT), repo_id=HF_REPO, path_in_repo="adapter")
    for path in GGUF_FILES.values():
        api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name, repo_id=HF_REPO)
    print(f"Uploaded to https://huggingface.co/{HF_REPO}")

## Files saved to Drive (`MyDrive/NutriAssist-RW/outputs/`)

| File | Contents |
|---|---|
| `experiment_table.csv`, `experiments.csv` | Settings, training time, peak GPU memory, losses and validation scores of every run |
| `final_comparison.csv` | Base vs fine-tuned on test and paraphrases, per language, with % improvement |
| `qualitative_test.csv` | Every test question with the reference, base and fine-tuned answers |
| `predictions/` | All generated answers and scores |
| `adapters/<experiment>/` | LoRA adapters, training logs and settings |
| `*.png` | The figures above, ready for the report |
| `nutriassist-rw-q4_k_m.gguf`, `nutriassist-rw-q8_0.gguf` | The fine-tuned model for CPU inference in the web app (put one in the project's `models/` folder) |